## Notebook 2: Models
Expanding window training and validation for all 3 models: softmax regression, XGBoost, and neural network
Standard scaling refit and transformed on every window increase
Compares model preformance to 3 baselines: random guessing, momentum, and reversion

In [1]:
# imports
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
import pickle
import warnings

In [2]:
warnings.filterwarnings('ignore')
retrain = True # feature flag to activate training and predicting

if retrain:
    # load data
    df = pd.read_csv('../data/raw_master.csv')
    df.drop('Date', inplace=True, axis=1)
    rows = len(df)
    cols = len(df.columns)
    
    # Encode labels
    le = LabelEncoder()
    le.fit(df['label'])
    
    # Standard scaler - centers at 0 (mean = 0) with std = 1
    scaler = StandardScaler()
    
    # Predicted Value vs Target Label for each model
    results = {
        'Softmax Regression': {'preds': [], 'trues': []},
        'XGBoost':            {'preds': [], 'trues': []},
        'MLP Classifier':     {'preds': [], 'trues': []}
    }
    
    # expanding window loop
    for i in range(30, rows):
        # models
        models = {
            'Softmax Regression' : LogisticRegression(
                solver='lbfgs',      # Supports multinomial classification (softmax)
                penalty='l2',        # Default regularization type (L2 regularization)
                C=1.0                # Inverse regularization strength
            ),
            
            'XGBoost' : XGBClassifier(
                eval_metric='logloss', # log loss function
                random_state=42,
                n_estimators=50,
                max_depth=3,
                n_jobs=1
            ),
            
            'MLP Classifier' : MLPClassifier(
                alpha=0.1,
                hidden_layer_sizes=(50,), # 1 hidden layer with 50 neurons
                max_iter=200
            )
        }
    
        X_train = df.iloc[:i, :-1]
        X_test = df.iloc[[i], :-1]
        
        y_train = le.transform(df.iloc[:i, -1])
        y_test = le.transform(df.iloc[[i], -1])
    
        scaler.fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)
    
        for name, model in models.items():
            model.fit(X_train_scaled, y_train)
            
            predicted = model.predict(X_test_scaled)
            
            results[name]['preds'].append(predicted)
            results[name]['trues'].append(y_test)
    
    with open('../data/results.pkl', 'wb') as f:
        pickle.dump(results, f)
else:
    with open('../data/results.pkl', 'rb') as f:
        results = pickle.load(f)